In [1]:
import pyspark
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .master("local[*]") \
    .appName('test') \
    .getOrCreate()

In [2]:
df_green = spark.read.parquet('data/raw/green/*/*')

In [3]:
df_green.registerTempTable('green')

C:\Users\adilb\AppData\Local\Programs\Python\Python311\Lib\site-packages\pyspark\sql\dataframe.py:329: FutureWarning: Deprecated in 2.0, use createOrReplaceTempView instead.
  warnings.warn("Deprecated in 2.0, use createOrReplaceTempView instead.", FutureWarning)


In [4]:
df_green.columns

['VendorID',
 'lpep_pickup_datetime',
 'lpep_dropoff_datetime',
 'store_and_fwd_flag',
 'RatecodeID',
 'PULocationID',
 'DOLocationID',
 'passenger_count',
 'trip_distance',
 'fare_amount',
 'extra',
 'mta_tax',
 'tip_amount',
 'tolls_amount',
 'ehail_fee',
 'improvement_surcharge',
 'total_amount',
 'payment_type',
 'trip_type',
 'congestion_surcharge']

In [5]:
df_green_revenue = spark.sql("""
SELECT 
    date_trunc('hour', lpep_pickup_datetime) AS hour, 
    PULocationID AS zone,

    SUM(total_amount) AS amount,
    COUNT(1) AS number_records
FROM
    green
WHERE
    lpep_pickup_datetime >= '2020-01-01 00:00:00'
GROUP BY
    1, 2
""")

In [6]:
df_green_revenue.show()

+-------------------+----+------------------+--------------+
|               hour|zone|            amount|number_records|
+-------------------+----+------------------+--------------+
|2020-01-01 00:00:00|  52|             83.33|             4|
|2020-01-01 00:00:00|  55|            129.29|             4|
|2020-01-01 02:00:00|  65|            176.89|             8|
|2020-01-01 06:00:00| 242|              6.96|             1|
|2020-01-01 10:00:00| 244|179.04999999999998|             9|
|2020-01-01 12:00:00| 179|               6.8|             1|
|2020-01-01 12:00:00| 112|             22.55|             1|
|2020-01-01 15:00:00| 169|              32.6|             2|
|2020-01-01 16:00:00| 193|             113.2|             4|
|2020-01-01 20:00:00|  25|            100.86|             8|
|2020-01-02 12:00:00| 241|              15.8|             1|
|2020-01-02 15:00:00| 131|151.70000000000002|             7|
|2020-01-02 19:00:00| 228|              29.8|             1|
|2020-01-02 23:00:00| 16

In [7]:
df_green_revenue \
    .repartition(20) \
    .write.parquet('data/report/revenue/green', mode='overwrite')

In [8]:
df_yellow = spark.read.parquet('data/raw/yellow/*/*')
df_yellow.registerTempTable('yellow')

In [9]:
df_yellow_revenue = spark.sql("""
SELECT 
    date_trunc('hour', tpep_pickup_datetime) AS hour, 
    PULocationID AS zone,

    SUM(total_amount) AS amount,
    COUNT(1) AS number_records
FROM
    yellow
WHERE
    tpep_pickup_datetime >= '2020-01-01 00:00:00'
GROUP BY
    1, 2
""")

In [10]:
df_yellow_revenue.show()

+-------------------+----+------------------+--------------+
|               hour|zone|            amount|number_records|
+-------------------+----+------------------+--------------+
|2020-01-01 00:00:00| 158| 4295.580000000011|           226|
|2020-01-01 00:00:00|  52|              49.8|             2|
|2020-01-01 01:00:00| 100| 1711.839999999999|            72|
|2020-01-01 02:00:00| 114| 2998.550000000002|           156|
|2020-01-01 02:00:00|  65|355.36000000000007|            18|
|2020-01-01 03:00:00| 164| 5656.480000000018|           312|
|2020-01-01 06:00:00|  24|              59.7|             4|
|2020-01-01 10:00:00| 244|            141.05|             7|
|2020-01-01 12:00:00|  48| 4731.730000000015|           270|
|2020-01-01 12:00:00| 179|108.25999999999999|             8|
|2020-01-01 12:00:00| 112|              19.3|             1|
|2020-01-01 14:00:00|  93|            205.75|             3|
|2020-01-01 16:00:00| 193|              40.4|             3|
|2020-01-01 20:00:00|  2

In [11]:
df_yellow_revenue \
    .repartition(20) \
    .write.parquet('data/report/revenue/yellow', mode='overwrite')

In [12]:
df_green_revenue = spark.read.parquet('data/report/revenue/green')
df_yellow_revenue = spark.read.parquet('data/report/revenue/yellow')

In [13]:
df_green_revenue_tmp = df_green_revenue \
    .withColumnRenamed('amount', 'green_amount') \
    .withColumnRenamed('number_records', 'green_number_records')

df_yellow_revenue_tmp = df_yellow_revenue \
    .withColumnRenamed('amount', 'yellow_amount') \
    .withColumnRenamed('number_records', 'yellow_number_records')

In [14]:
df_join = df_green_revenue_tmp.join(df_yellow_revenue_tmp, on=['hour', 'zone'], how='outer')

In [15]:
df_join.show()

+-------------------+----+------------------+--------------------+------------------+---------------------+
|               hour|zone|      green_amount|green_number_records|     yellow_amount|yellow_number_records|
+-------------------+----+------------------+--------------------+------------------+---------------------+
|2020-01-01 00:00:00|  50|              NULL|                NULL|  4177.48000000001|                  183|
|2020-01-01 00:00:00|  52|             83.33|                   4|              49.8|                    2|
|2020-01-01 00:00:00|  55|            129.29|                   4|              NULL|                 NULL|
|2020-01-01 00:00:00|  69|              11.8|                   1|106.52000000000001|                    4|
|2020-01-01 00:00:00| 106|             10.56|                   1|              NULL|                 NULL|
|2020-01-01 00:00:00| 112|312.26000000000005|                  18|119.47999999999999|                    8|
|2020-01-01 00:00:00| 127|  

In [16]:
df_join.write.parquet('data/report/revenue/total', mode='overwrite')

In [17]:
df_join = spark.read.parquet('data/report/revenue/total')

In [18]:
df_join.show()

+-------------------+----+------------------+--------------------+------------------+---------------------+
|               hour|zone|      green_amount|green_number_records|     yellow_amount|yellow_number_records|
+-------------------+----+------------------+--------------------+------------------+---------------------+
|2020-01-01 00:00:00| 135|              NULL|                NULL|              18.3|                    1|
|2020-01-01 00:00:00| 191|             84.21|                   2|              57.3|                    1|
|2020-01-01 00:00:00| 211|              NULL|                NULL|2412.2599999999993|                  116|
|2020-01-01 00:00:00| 232|              NULL|                NULL|1124.9399999999996|                   51|
|2020-01-01 00:00:00| 265|              NULL|                NULL|           1041.52|                   15|
|2020-01-01 01:00:00|  36|            192.41|                   9|            519.38|                   20|
|2020-01-01 01:00:00|  49|23

In [19]:
df_zones = spark.read.parquet('zones/')

In [20]:
df_zones.show()

+----------+-------------+--------------------+------------+
|LocationID|      Borough|                Zone|service_zone|
+----------+-------------+--------------------+------------+
|         1|          EWR|      Newark Airport|         EWR|
|         2|       Queens|         Jamaica Bay|   Boro Zone|
|         3|        Bronx|Allerton/Pelham G...|   Boro Zone|
|         4|    Manhattan|       Alphabet City| Yellow Zone|
|         5|Staten Island|       Arden Heights|   Boro Zone|
|         6|Staten Island|Arrochar/Fort Wad...|   Boro Zone|
|         7|       Queens|             Astoria|   Boro Zone|
|         8|       Queens|        Astoria Park|   Boro Zone|
|         9|       Queens|          Auburndale|   Boro Zone|
|        10|       Queens|        Baisley Park|   Boro Zone|
|        11|     Brooklyn|          Bath Beach|   Boro Zone|
|        12|    Manhattan|        Battery Park| Yellow Zone|
|        13|    Manhattan|   Battery Park City| Yellow Zone|
|        14|     Brookly

In [21]:
df_result = df_join.join(df_zones, df_join.zone == df_zones.LocationID)

In [22]:
df_result.show()

+-------------------+----+------------------+--------------------+------------------+---------------------+----------+---------+--------------------+------------+
|               hour|zone|      green_amount|green_number_records|     yellow_amount|yellow_number_records|LocationID|  Borough|                Zone|service_zone|
+-------------------+----+------------------+--------------------+------------------+---------------------+----------+---------+--------------------+------------+
|2020-01-01 00:00:00| 135|              NULL|                NULL|              18.3|                    1|       135|   Queens|   Kew Gardens Hills|   Boro Zone|
|2020-01-01 00:00:00| 191|             84.21|                   2|              57.3|                    1|       191|   Queens|      Queens Village|   Boro Zone|
|2020-01-01 00:00:00| 211|              NULL|                NULL|2412.2599999999993|                  116|       211|Manhattan|                SoHo| Yellow Zone|
|2020-01-01 00:00:00| 

In [23]:
df_result.drop('LocationID', 'zone').write.parquet('tmp/revenue-zones')

In [24]:
df_result = spark.read.parquet('tmp/revenue-zones')

In [25]:
df_result.show()

+-------------------+------------------+--------------------+------------------+---------------------+---------+------------+
|               hour|      green_amount|green_number_records|     yellow_amount|yellow_number_records|  Borough|service_zone|
+-------------------+------------------+--------------------+------------------+---------------------+---------+------------+
|2020-01-01 00:00:00|              NULL|                NULL|              25.0|                    1|    Bronx|   Boro Zone|
|2020-01-01 00:00:00|            107.52|                   6| 6539.510000000026|                  390|Manhattan| Yellow Zone|
|2020-01-01 00:00:00|              NULL|                NULL|10773.359999999973|                  455|Manhattan| Yellow Zone|
|2020-01-01 00:00:00|             34.46|                   2|              NULL|                 NULL|    Bronx|   Boro Zone|
|2020-01-01 00:00:00|              NULL|                NULL| 823.7999999999997|                   36|Manhattan| Yello